# 01 · Floods — water segmentation (Sen1Floods11)

**CRISP module:** Floods · **Task:** per-pixel flood-water segmentation · **Model:** U-Net / ResNet-34

Trains three models on the hand-labelled Sen1Floods11 chips and compares them:

| Model | Input | Why |
|---|---|---|
| `flood_s1` | Sentinel-1 radar VV, VH | Sees through cloud — floods happen under storm cloud |
| `flood_s2` | Sentinel-2, 12 bands | Best accuracy when the sky is clear |
| `flood_s1s2` | both | Fusion |

Each is scored on the official **test** split and on **Bolivia**, a flood event never seen in training
(geographic generalisation), and compared with the index-threshold rules CRISP uses today.
Runtime on a T4: about 25–35 min for all three.

In [ ]:
# ── Setup: run once per Colab session (Runtime ▸ Change runtime type ▸ T4 GPU) ──
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/inayatarshad/trainedmodels_gstp.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content")
    DRIVE = Path("/content/drive/MyDrive/gstp_runs")       # checkpoints + results persist here
    REPO = WORK / "trainedmodels_gstp"
    if not REPO.exists():
        if subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)]).returncode != 0:
            # Private repo? Upload the repo zip to MyDrive/trainedmodels_gstp.zip and re-run.
            shutil.unpack_archive("/content/drive/MyDrive/trainedmodels_gstp.zip", WORK)
            REPO = next(WORK.glob("trainedmodels_gstp*"))
    os.chdir(REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:  # local / Kaggle
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(REPO)
    WORK = DRIVE = REPO / "runs"

sys.path.insert(0, str(REPO / "src"))
DRIVE.mkdir(parents=True, exist_ok=True)

import torch
from gstp.utils import seed_everything, get_device
seed_everything(42)
print("Repo:", REPO, "| device:", get_device(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
MODULE = "floods"
from gstp.data import sen1floods11 as S
from gstp.data.segmentation import SegDataset
from gstp.data.raster import compute_band_stats, read_raster
from gstp.models import build_unet, count_parameters
from gstp import train_seg, baselines, viz
from gstp.utils import save_json, load_json
import numpy as np, pandas as pd, torch

DATA = WORK / "data" / "sen1floods11"
RESULTS = REPO / "results" / MODULE
CKPT_DIR = DRIVE / MODULE
for p in (DATA, RESULTS / "figures", CKPT_DIR):
    p.mkdir(parents=True, exist_ok=True)

## 1. Download the hand-labelled chips (~3 GB, public Google Cloud bucket)

In [ ]:
if not any(DATA.rglob("*_LabelHand.tif")):
    for sub in ("S1Hand", "S2Hand", "LabelHand"):
        subprocess.run(["gsutil", "-m", "-q", "cp", "-r",
                        f"gs://sen1floods11/v1.1/data/flood_events/HandLabeled/{sub}", str(DATA)], check=True)
    subprocess.run(["gsutil", "-m", "-q", "cp", "-r", "gs://sen1floods11/v1.1/splits", str(DATA)], check=True)
print({d: len(list(DATA.rglob(f"*_{d}.tif"))) for d in ("S1Hand", "S2Hand", "LabelHand")})
print([p.name for p in DATA.rglob("*.csv")])

## 2. Build the splits and inspect the data

In [ ]:
splits = {m: {sp: S.build_samples(DATA, sp, m) for sp in ("train", "val", "test", "bolivia")} for m in S.MODALITIES}
print({sp: len(v) for sp, v in splits["s1s2"].items()})
assert all(len(v) for v in splits["s1s2"].values()), "An empty split — check the downloaded CSVs"
events = pd.Series([s.meta["event"] for s in splits["s1s2"]["train"]]).value_counts()
print("Training chips per flood event:\n", events.to_string())

In [ ]:
# Training-set statistics (never computed on val/test) for z-score normalisation
from gstp.data.segmentation import read_sample_image
stats = {m: compute_band_stats([s for s in splits[m]["train"]], read_sample_image) for m in S.MODALITIES}
for m in stats:
    print(m, "bands:", S.MODALITIES[m]["bands"])
    print("   mean:", np.round(stats[m]["mean"], 2))
save_json(stats, RESULTS / "band_stats.json")

## 3. Train the three models

In [ ]:
EPOCHS = 60          # early stopping usually ends earlier
summaries, models = {}, {}
for m in ("s2", "s1", "s1s2"):
    seed_everything(42)
    bands = S.MODALITIES[m]["bands"]
    cfg = train_seg.SegTrainConfig(name=f"flood_{m}", in_channels=len(bands), bands=bands, epochs=EPOCHS,
                                   batch_size=16, crop_size=256, lr=3e-4, patience=12,
                                   extra={"norm": "zscore", "stats": stats[m], "modality": m,
                                          "units": "S1: dB; S2: TOA reflectance x 10000", "dataset": "Sen1Floods11 v1.1"})
    tr = SegDataset(splits[m]["train"], "zscore", stats[m], train=True, crop_size=256)
    va = SegDataset(splits[m]["val"], "zscore", stats[m])
    model = build_unet(len(bands))
    print(f"\n=== flood_{m}: {len(bands)} bands, {count_parameters(model)/1e6:.1f} M parameters ===")
    summaries[m] = train_seg.train(model, tr, va, cfg, CKPT_DIR)
    models[m] = model
    viz.training_curves(summaries[m]["history"], title=f"flood_{m}", path=RESULTS / "figures" / f"curves_flood_{m}.png")

## 4. Test once — official test split and the unseen Bolivia event

In [ ]:
reports = {}
for m, model in models.items():
    thr = summaries[m]["chosen_threshold"]
    tests = {sp: SegDataset(splits[m][sp], "zscore", stats[m]) for sp in ("test", "bolivia")}
    reports[m] = train_seg.test_report(model, tests, thr, RESULTS / f"test_flood_{m}.json",
                                       group_key=lambda s: s.meta["event"])
rows = []
for m, r in reports.items():
    for sp in ("test", "bolivia"):
        x = r[sp]["at_chosen_threshold"]
        rows.append({"model": f"U-Net {m}", "split": sp, "IoU (water)": x["iou"], "F1": x["f1"],
                     "precision": x["precision"], "recall": x["recall"], "mIoU": x["miou"]})

## 5. Baselines: the index thresholds CRISP-style rules use

In [ ]:
s2_bands = S.MODALITIES["s2"]["bands"]
G, SW1 = s2_bands.index("B3"), s2_bands.index("B11")
bl = {
    "MNDWI threshold (S2)": baselines.tune_and_test(
        splits["s2"]["val"], {"test": splits["s2"]["test"], "bolivia": splits["s2"]["bolivia"]},
        lambda x: baselines.normalized_difference(x[G], x[SW1]), ">"),
    "VV backscatter threshold (S1)": baselines.tune_and_test(
        splits["s1"]["val"], {"test": splits["s1"]["test"], "bolivia": splits["s1"]["bolivia"]},
        lambda x: x[0], "<"),
}
for name, r in bl.items():
    for sp in ("test", "bolivia"):
        x = r[sp]
        rows.append({"model": name, "split": sp, "IoU (water)": x["iou"], "F1": x["f1"],
                     "precision": x["precision"], "recall": x["recall"], "mIoU": x["miou"]})
save_json(bl, RESULTS / "baselines.json")
table = pd.DataFrame(rows).sort_values(["split", "IoU (water)"], ascending=[True, False])
table.to_csv(RESULTS / "results_table.csv", index=False)
table.style.format(precision=3).hide(axis="index")

## 6. Figures — threshold choice, per-event scores and example predictions

In [ ]:
from gstp.data.raster import IGNORE_INDEX
best_m = max(reports, key=lambda m: reports[m]["test"]["at_chosen_threshold"]["iou"])
print("Best model on test:", best_m)
val_meter, _ = train_seg.evaluate(models[best_m], SegDataset(splits[best_m]["val"], "zscore", stats[best_m]))
viz.threshold_curve(val_meter, summaries[best_m]["chosen_threshold"], f"flood_{best_m}: validation",
                    RESULTS / "figures" / "threshold_curve.png")

per_event = pd.DataFrame({m: {g: v["iou"] for g, v in reports[m]["test"]["per_group"].items()} for m in reports})
per_event.to_csv(RESULTS / "per_event_iou.csv")
ax = per_event.plot.bar(figsize=(9, 3.5), title="Test IoU per flood event"); ax.set_ylabel("IoU")
ax.figure.tight_layout(); ax.figure.savefig(RESULTS / "figures" / "per_event_iou.png", dpi=150)

In [ ]:
def examples(split, m, n=4):
    ds = SegDataset(splits[m][split], "zscore", stats[m])
    raw = SegDataset(splits["s2"][split], "none")
    model = models[m].eval()
    order = np.argsort([-np.mean(ds.load(i)[1] == 1) for i in range(len(ds))])[: n * 3 : 3]  # flood-rich chips
    out = []
    for i in order:
        x, y = ds.load(i)
        with torch.no_grad():
            p = torch.sigmoid(model(torch.from_numpy(x)[None].to(get_device()))).squeeze().cpu().numpy()
        img, _ = raw.load(i)
        out.append({"rgb": viz.rgb(img, (3, 2, 1)), "mask": y, "prob": p,
                    "thr": summaries[m]["chosen_threshold"], "name": ds.samples[i].id})
    return out
for sp in ("test", "bolivia"):
    viz.prediction_grid(examples(sp, best_m), f"flood_{best_m} on {sp}", RESULTS / "figures" / f"examples_{sp}.png")

## 7. Export: ONNX for the later Jetson / TensorRT step, plus a results summary

In [ ]:
from gstp.models import export_onnx
for m, model in models.items():
    export_onnx(model, len(S.MODALITIES[m]["bands"]), 512, str(CKPT_DIR / f"flood_{m}.onnx"))
save_json({"summaries": {m: {k: v for k, v in s.items() if k != "history"} for m, s in summaries.items()},
           "test": reports}, RESULTS / "summary.json")

In [ ]:
# ── Package results: metrics + figures go to Drive as a zip you can download ──
import shutil
archive = shutil.make_archive(str(DRIVE / f"results_{MODULE}"), "zip", RESULTS)
print("Results zip :", archive)
print("Checkpoints :", sorted(str(p) for p in CKPT_DIR.glob("*.pt")))